In [4]:
from glob import glob 

for g in glob('./data/*.pdf'):
    print(g)

./data\2040_seoul_plan.pdf
./data\OneNYC_2050_Strategic_Plan.pdf


In [5]:
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


def read_pdf_and_split_text(pdf_path, chunk_size=1000, chunk_overlap=100):
    print(f"PDF: {pdf_path} -----------------------------")

    reader = PdfReader(pdf_path)

    # 각 페이지의 텍스트를 LangChain Document로 변환
    documents = []
    for page_index, page in enumerate(reader.pages):
        text = page.extract_text() or ""

        if text.strip():
            documents.append(
                Document(
                    page_content=text,
                    metadata={
                        "source": str(pdf_path),
                        "page": page_index,  # 0부터 시작
                    },
                )
            )

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )

    splits = text_splitter.split_documents(documents)

    print(f"Number of splits: {len(splits)}\n")
    return splits

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3",
    model_kwargs = {'device': 'cuda'}, # cuda 지원하면 cuda, 아닌 경우 cpu
    encode_kwargs = {'normalize_embeddings': True}, # 임베딩 정규화
)

W1101 21:27:55.650000 28008 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [7]:
embeddings.embed_documents("안녕하세요")

[[0.0024569316301494837,
  0.03226260095834732,
  -0.007424218114465475,
  0.00526838144287467,
  -0.058097343891859055,
  -0.030428627505898476,
  -0.00830004271119833,
  0.03674236312508583,
  0.009622698649764061,
  -0.007633987348526716,
  0.017204491421580315,
  0.038877006620168686,
  -0.029649028554558754,
  -0.02181999199092388,
  -0.0008699322352185845,
  -0.032562460750341415,
  0.03178016468882561,
  -0.022462934255599976,
  0.023502551019191742,
  -0.012333737686276436,
  -0.03881021589040756,
  -0.017907658591866493,
  0.03872933238744736,
  0.013715693727135658,
  0.025562569499015808,
  0.022995948791503906,
  -0.027591779828071594,
  0.0270734503865242,
  -0.009775781072676182,
  -0.026509873569011688,
  -0.006837868131697178,
  -0.0293024480342865,
  0.0288232434540987,
  -0.07535597681999207,
  -0.03274381905794144,
  -0.004341182764619589,
  -0.02311619557440281,
  0.02154519222676754,
  -0.05429789796471596,
  0.052812572568655014,
  0.04333137720823288,
  -0.020881

In [9]:
from pypdf import overwrite_configuration

overwrite_configuration(
    stream_parsing_work_maximum_count=10_000_000
)

Configuration(maximum_declared_stream_length=75000000, array_based_stream_maximum_output_length=75000000, array_maximum_element_count=100000, dictionary_maximum_element_count=100000, jbig2_maximum_output_length=75000000, lzw_maximum_output_length=75000000, run_length_maximum_output_length=75000000, zlib_maximum_output_length=75000000, zlib_maximum_recovery_input_length=5000000, brotli_maximum_output_length=75000000, flate_maximum_columns=250000, flate_maximum_row_length=4000000, image_maximum_buffer_size=75000000, stream_filters_maximum_length=16, stream_decoding_work_maximum_length=300000000, stream_parsing_work_maximum_count=10000000, xmp_maximum_input_length=5000000, xmp_maximum_element_count=100000, outline_maximum_entries=100000, outline_maximum_depth=100, page_tree_maximum_entries=100000, page_tree_maximum_depth=100, page_tree_maximum_page_entries=2000000, form_fields_maximum_entries=100000, form_fields_maximum_depth=100, xform_maximum_invocations_per_extraction=5000, jbig2dec_bi

In [ ]:
from langchain_chroma import Chroma

import os
persist_directory='./chroma_store'

if os.path.exists(persist_directory):
    print("Loading existing Chroma store")
    vectorstore = Chroma(
        persist_directory=persist_directory, 
        embedding_function=embeddings
    )
else:
    print("Creating new Chroma store")
    
    all_splits = []
    for g in glob('./data/*.pdf'):
        all_splits.extend(read_pdf_and_split_text(g))

    print(f"Total number of splits: {len(all_splits)}")

    vectorstore = Chroma.from_documents(
        documents=all_splits,
        embedding=embeddings,
        persist_directory=persist_directory
    )

Creating new Chroma store
PDF: ./data\2040_seoul_plan.pdf -----------------------------
Number of splits: 288

PDF: ./data\OneNYC_2050_Strategic_Plan.pdf -----------------------------


fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/SLGUZI+DINNextLTPro-Bold', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(5397, 0, 1386391892720), '/LastChar': 110, '/Subtype': '/Type1', '/Type': '/Font', '/Widths': [236, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 556, 556, 556, 0, 0, 556, 0, 0, 0, 556, 0, 0, 0, 0, 0, 0, 0, 644, 610, 575, 630, 565, 557, 628, 658, 277, 0, 645, 550, 763, 655, 628, 605, 628, 624, 595, 557, 645, 595, 865, 595, 568, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 527, 0, 0, 0, 0, 0, 0, 0, 0, 553]}, but is not installed. Consider installing fontTools if you encounter encoding problems.
fontTools is required to fully parse the encoding of a CFF Type1 font in font dictionary {'/BaseFont': '/WHFNDA+DINNextLTPro-Medium', '/Encoding': '/WinAnsiEncoding', '/FirstChar': 32, '/FontDescriptor': IndirectObject(4170, 0, 1386391892720), '/LastChar': 86, '/Subtype': '/Type1', '/Type': '/Fon

Number of splits: 1022

Total number of splits: 1310


In [11]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

chunks = retriever.invoke("서울시 쓰레기 저감 정책")

for chunk in chunks:
    print(chunk.metadata)
    print(chunk.page_content)

{'page': 64, 'source': './data\\2040_seoul_plan.pdf'}
제3절 2040 서울도시기본계획 7대 목표 57
Ÿ 특히, 서울시는 현재 온실가스 배출량의 90%를 차지하고 있는 건물과 수송 부문 감축을 위해 적극
적인 대책을 마련하고 있다.
- 2026년까지 건물 에너지효율화사업 100만 호를 추진, 건물온실가스총량제, 신규건물 제로에너지 
건물(ZEB) 의무화를 통해 기존 및 신규건물의 제로에너지화를 촉진
- 수송 부문 배출감축을 위해 전기차 비중을 2026년까지 10%(’21년 5.2만 대 → ’26년 40만 대)로 확
대하고, 22만기의 충전 인프라 구축을 계획 중(’21년 2만기) 
시민 생활과 안전을 위한 기후위기 대응 요구 심화
Ÿ 서울은 인구, 시설 등이 밀집해 있는 대도시로 기온 상승, 폭염, 집중호우, 태풍, 한파 등 극한 기후 
현상이 더욱 빈번하게 발생할 것으로 전망된다. 이러한 기후위험은 서울시민의 일상생활과 안전을 
크게 위협할 수 있다. 따라서 시민의 일상을 보호하고 도시의 회복력을 강화하기 위한 적극적인 기
후위기 대응 전략이 필요하다.
Ÿ 특히 다양화·복합화되는 재난안전사고에 대응해 전통적인 자연·사회재난의 범주뿐 아니라, 신종 
복합재난까지 대비할 수 있는 다면적인 대응체계 마련이 요구된다.
2) 추진전략
탄소중립·기후위기 적응대책은 시의 모든 정책과 사업의 주요 원칙으로 고려
Ÿ 탄소중립은 돌이키기 어려운 기후재난을 막기 위한 국제사회와의 약속이기에 시의 모든 정책과 
사업의 결정 과정에서 주요하게 고려해야 할 포괄적 ‘원칙’으로 확립되어야 한다.
- 서울시는 온실가스 감축 방안을 모든 예산사업에 고려·반영하기 위해 2021년부터 기후예산제 시범
사업을 진행 중
Ÿ 모든 실·국·본부와 관련 이해 당사자들은 이러한 원칙과 기본 방향에 적극적으로 참여할 필요가 있으며 
기후변화로 인한 폭염, 홍수, 가뭄 등과 같은 기상재해는 향후 지속적으로 발생할 가능성이 높아, 
정책 수립 시 주요하게 고려한다.